In [2]:
from pathlib import Path
import pandas as pd

carpeta  = Path(r'source\matricula_y_secciones')

dfs = []

for archivo in sorted(carpeta.glob('*.csv')):
    df = pd.read_csv(archivo, sep=";",encoding='utf-8')

    df.columns = df.columns.str.strip().str.lower()

    df["anio"] = int(archivo.name[:4])

    dfs.append(df)

In [4]:
# Paso 3: comparar columnas entre archivos

# Armamos un diccionario con las columnas de cada archivo
columnas_por_archivo = {}
for f in sorted(carpeta.glob("*.csv")):
    df = pd.read_csv(f, sep=None, engine="python", encoding="utf-8", encoding_errors="replace")
    df.columns = (df.columns
                  .str.strip()
                  .str.lower()
                  .str.replace("\ufeff", "",regex=False)
                  )
    columnas_por_archivo[f.name[:4]] = set(df.columns)

# Tomamos el primer año como referencia
anio_ref = list(columnas_por_archivo.keys())[0]
cols_ref = columnas_por_archivo[anio_ref]

print(f"Referencia: {anio_ref} → {sorted(cols_ref)}\n")

for anio, cols in columnas_por_archivo.items():
    if anio == anio_ref:
        continue
    faltantes = cols_ref - cols
    extras    = cols - cols_ref
    if not faltantes and not extras:
        print(f"✔  {anio}  → igual")
    else:
        print(f"✘  {anio}")
        if faltantes: print(f"     Faltan:  {sorted(faltantes)}")
        if extras:    print(f"     Sobran:  {sorted(extras)}")

Referencia: 2011 → ['_1', '_10', '_11', '_12', '_1314', '_2', '_3', '_4', '_5', '_6', '_7', '_8', '_9', '_sec1', '_sec10', '_sec11', '_sec12', '_sec1314', '_sec2', '_sec3', '_sec4', '_sec5', '_sec6', '_sec7', '_sec8', '_sec9', '_secdeambu', '_seclactantes', '_secs2', '_secs3', '_secs4', '_secs5', '_snu', 'ambito', 'deambulantes', 'departamento', 'lactantes', 'multi_ini', 'multi_pri', 'multi_sec', 'multinivel', 'provincia', 'r_1', 'r_10', 'r_11', 'r_12', 'r_1314', 'r_2', 'r_3', 'r_4', 'r_5', 'r_6', 'r_7', 'r_8', 'r_9', 's2', 's3', 's4', 's5', 's_1', 's_10', 's_11', 's_12', 's_1314', 's_2', 's_3', 's_4', 's_5', 's_6', 's_7', 's_8', 's_9', 's_deambulantes', 's_lactantes', 's_s2', 's_s3', 's_s4', 's_s5', 'sector', 'v_1', 'v_10', 'v_11', 'v_12', 'v_1314', 'v_2', 'v_3', 'v_4', 'v_5', 'v_6', 'v_7', 'v_8', 'v_9', 'v_deambulantes', 'v_lactantes', 'v_s2', 'v_s3', 'v_s4', 'v_s5', 'v_snu']

✔  2012  → igual
✔  2013  → igual
✔  2014  → igual
✔  2015  → igual
✔  2016  → igual
✔  2017  → igual
✔  201

In [6]:
dfs = []
for f in sorted(carpeta.glob("*.csv")):
    
    # cargar
    df = pd.read_csv(f, sep=None, engine="python", encoding="utf-8")
    
    # normalizar + eliminar BOM
    df.columns = (df.columns
                  .str.strip()
                  .str.lower()
                  .str.replace("\ufeff", "", regex=False))
    
    # extraer año del nombre del archivo
    df["anio"] = int(f.name[:4])
    
    # borramos las columnas que sobran en los dos ultimos archivos
    cols_sobran = ["_20", "_sec20", "r_20", "v_20", "localizacion"]
    df = df.drop(columns=[c for c in cols_sobran if c in df.columns])
    
    dfs.append(df)

# concatenar
df_final = pd.concat(dfs, ignore_index=True)

# anio primera columna
cols = ["anio"] + [c for c in df_final.columns if c != "anio"]
df_final = df_final[cols]

df_final.to_csv(r"data\matricula_y_secciones_final.csv", index=False, encoding="utf-8-sig")

print("Archivo exportado!")

print(df_final.shape)
print(df_final.head())

Archivo exportado!
(16803, 100)
   anio     provincia departamento   sector  ambito lactantes deambulantes  \
0  2011  Buenos Aires   25 DE MAYO  Estatal   Rural                          
1  2011  Buenos Aires   25 DE MAYO  Estatal  Urbano         8           26   
2  2011  Buenos Aires   25 DE MAYO  Privado  Urbano         8           13   
3  2011  Buenos Aires   9 DE JULIO  Estatal   Rural                          
4  2011  Buenos Aires   9 DE JULIO  Estatal  Urbano                          

    s2   s3   s4  ... _sec8 _sec9 _sec10 _sec11 _sec12 _sec1314 multi_ini  \
0   63   82   90  ...     6     6      6      6      6        0        12   
1  127  330  377  ...    16    14     12     12      8        0        19   
2   24  128  136  ...     4     4      3      3      3        0         3   
3   30   96  125  ...     7     6      4      4      4        0        20   
4   26  471  512  ...    23    19     22     21     18        0        19   

  multi_pri multi_sec multinivel  
0